# 04 — ML Return Prediction Models

Extends the factor-based approach to machine learning models for
cross-sectional return prediction.

## Models

| # | Model | Key Idea |
|---|-------|----------|
| 1 | Linear Regression | Baseline |
| 2 | RBF Kernel Ridge | Nystroem approximation + Ridge for nonlinear features |
| 3 | Random Forest | Ensemble of shallow trees |
| 4 | Deep Neural Network | Keras NN with L1 regularization + coarse-to-fine search |
| 5 | AdaBoost | Boosted decision stumps with coarse-to-fine grid search |
| 6 | Max Sharpe Regression | Custom PyTorch loss that maximizes portfolio Sharpe |
| 7 | IPCA | Instrumented PCA — latent factor model |

## Evaluation

Each model is evaluated on:
- OOS MSE, R², sign accuracy
- Long/short decile spread portfolio (Sharpe, drawdown, hit rate)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gc
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression

from quant_trading.data import (
    load_jkp_csv, winsorize_returns, prepare_ml_features,
    TARGET_COL, NUMERIC_FEATURES_ML, CATEGORICAL_FEATURES,
)
from quant_trading.signals import generate_positions
from quant_trading.portfolio import calculate_portfolio_returns
from quant_trading.evaluation import (
    evaluate_model_performance, calculate_max_drawdown,
    oos_r2, sort_ret_eq_wgt,
)

## 1. Data Loading & Preprocessing

In [ ]:
JKP_CSV_PATH = "../data/jkp_data.csv"
df_jkp = load_jkp_csv(JKP_CSV_PATH)
df_jkp = winsorize_returns(df_jkp, lower=0.05, upper=0.95)
df_jkp = df_jkp.loc[:, ~df_jkp.columns.duplicated(keep="first")]

X_all, meta, y_all = prepare_ml_features(
    df_jkp,
    numeric_features=NUMERIC_FEATURES_ML,
    categorical_features=CATEGORICAL_FEATURES,
    target_col=TARGET_COL,
)

features = list(X_all.columns)
print(f"Total rows: {len(X_all):,} | Features: {len(features)}")

In [ ]:
# Train/test split
TRAIN_END_DATE = "2015-12-31"
TEST_START_DATE = "2016-01-01"

train_mask = meta["month_date"] <= pd.Timestamp(TRAIN_END_DATE)
test_mask = meta["month_date"] >= pd.Timestamp(TEST_START_DATE)

X_train, X_test = X_all.loc[train_mask], X_all.loc[test_mask]
y_train, y_test = y_all.loc[train_mask], y_all.loc[test_mask]

# Build test_df for portfolio evaluation (need id, month_date, target, market_equity)
test_df = meta.loc[test_mask].copy()
test_df = test_df.merge(
    df_jkp[["id", "month_date", "market_equity"]].drop_duplicates(),
    on=["id", "month_date"], how="left",
)

print(f"Train: {len(X_train):,} rows | Test: {len(X_test):,} rows")
print(f"Train period: {meta.loc[train_mask, 'month_date'].min().date()} to {meta.loc[train_mask, 'month_date'].max().date()}")
print(f"Test period:  {meta.loc[test_mask, 'month_date'].min().date()} to {meta.loc[test_mask, 'month_date'].max().date()}")

## Portfolio evaluation helper

In [ ]:
def evaluate_portfolio(df_eval, pred_col):
    """Evaluate long/short portfolio performance across percentile/weight grids."""
    pct_pairs = [(0.3, 0.7), (0.1, 0.9), (0.05, 0.95), (0.01, 0.99)]
    weight_schemes = ["equal", "char_rank_weighted", "char_minmax_weighted"]
    results = []

    for lower, upper in pct_pairs:
        positions = generate_positions(
            df_eval, pred_col, rebal_period=1,
            lower_pct=lower, upper_pct=upper, long_high=True,
        )
        for scheme in weight_schemes:
            try:
                port_rets = calculate_portfolio_returns(
                    positions, df_eval, ret_col=TARGET_COL,
                    weight_scheme=scheme, weight_col="market_equity",
                    char_col=pred_col, max_weight_per_leg=None,
                )
                spreads = port_rets["Spread"].dropna()
                if len(spreads) == 0:
                    continue
                ann_ret = spreads.mean() * 12
                ann_vol = spreads.std() * np.sqrt(12)
                results.append({
                    "Lower Pct": lower,
                    "Upper Pct": upper,
                    "Weight Scheme": scheme,
                    "Ann Ret (%)": ann_ret * 100,
                    "Ann Vol (%)": ann_vol * 100,
                    "Max DD (%)": calculate_max_drawdown(spreads) * 100,
                    "Sharpe": ann_ret / ann_vol if ann_vol > 0 else 0,
                    "Hit Rate (%)": (spreads > 0).mean() * 100,
                })
            except Exception as e:
                print(f"Error: {lower}-{upper} {scheme}: {e}")

    res_df = pd.DataFrame(results)
    print(f"\nPortfolio Evaluation for {pred_col}:")
    display(res_df.round(2))
    return res_df

## 2. Model 1 — Linear Regression (Baseline)

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
test_df["pred_lr"] = lr_model.predict(X_test)

_ = evaluate_model_performance(test_df[TARGET_COL], test_df["pred_lr"], "Linear Regression")
_ = evaluate_portfolio(test_df, "pred_lr")

## 3. Model 2 — RBF Kernel Ridge Regression

In [ ]:
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

rbf_ridge = Pipeline([
    ("nystroem", Nystroem(kernel="rbf", gamma=0.1, n_components=300, random_state=42)),
    ("ridge", Ridge(alpha=1.0)),
])

param_grid = {"ridge__alpha": [40.0, 50.0, 60.0]}
tscv = TimeSeriesSplit(n_splits=3)
search = GridSearchCV(rbf_ridge, param_grid, cv=tscv, scoring="neg_mean_squared_error")

print("Training RBF Kernel Ridge...")
search.fit(X_train, y_train)
print(f"Best alpha: {search.best_params_['ridge__alpha']}")

test_df["pred_rbf"] = search.predict(X_test)
_ = evaluate_model_performance(test_df[TARGET_COL], test_df["pred_rbf"], "RBF Kernel Ridge")
_ = evaluate_portfolio(test_df, "pred_rbf")

## 4. Model 3 — Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators=100, max_depth=6, n_jobs=-1, random_state=42)
print("Training Random Forest...")
rf_model.fit(X_train, y_train)

test_df["pred_rf"] = rf_model.predict(X_test)
_ = evaluate_model_performance(test_df[TARGET_COL], test_df["pred_rf"], "Random Forest")
_ = evaluate_portfolio(test_df, "pred_rf")

## 5. Model 4 — Deep Neural Network (Keras)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
import tensorflow as tf
import tensorflow.keras as keras
from tensorflow.keras.layers import Dense, BatchNormalization, Input
from tensorflow.keras import regularizers

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)


def build_nn(l1_reg):
    keras.backend.clear_session()
    model = keras.models.Sequential([
        Input(shape=(X_train_scaled.shape[1],)),
        BatchNormalization(),
        Dense(32, activation="relu",
              kernel_regularizer=regularizers.L1(l1=l1_reg),
              bias_regularizer=regularizers.L1(l1=1e-05)),
        BatchNormalization(),
        Dense(16, activation="relu",
              kernel_regularizer=regularizers.L1(l1=l1_reg),
              bias_regularizer=regularizers.L1(l1=1e-05)),
        BatchNormalization(),
        Dense(1, activation="linear"),
    ])
    model.compile(optimizer="adam", loss="mse")
    return model


def cv_search_l1(l1_list):
    best_l1, best_val_loss = l1_list[0], float("inf")
    tscv = TimeSeriesSplit(n_splits=2)
    for l1 in l1_list:
        print(f"  Testing L1 = {l1:.5f}")
        cv_losses = []
        for train_idx, val_idx in tscv.split(X_train_scaled):
            X_tr, X_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
            y_tr = y_train.iloc[train_idx].values.astype(np.float32)
            y_val = y_train.iloc[val_idx].values.astype(np.float32)
            with tf.device("/CPU:0"):
                model = build_nn(l1)
                cb = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3)
                history = model.fit(
                    X_tr, y_tr, epochs=10, batch_size=2048,
                    validation_data=(X_val, y_val), callbacks=[cb], verbose=0,
                )
                cv_losses.append(min(history.history["val_loss"]))
            keras.backend.clear_session()
            del model, history
            gc.collect()
        avg_loss = np.mean(cv_losses)
        print(f"  -> Avg Val Loss: {avg_loss:.6f}")
        if avg_loss < best_val_loss:
            best_val_loss = avg_loss
            best_l1 = l1
    return best_l1, best_val_loss


print("Stage 1: Coarse search...")
best_coarse_l1, _ = cv_search_l1([1e-5, 1e-4, 1e-3, 1e-2, 1e-1])
print(f"Best coarse L1: {best_coarse_l1:.5f}\n")

print("Stage 2: Fine search...")
best_fine_l1, _ = cv_search_l1([best_coarse_l1 * 0.5, best_coarse_l1, best_coarse_l1 * 2.0])
print(f"Best fine L1: {best_fine_l1:.5f}\n")

print("Training final model...")
with tf.device("/CPU:0"):
    final_nn = build_nn(best_fine_l1)
    cb = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    final_nn.fit(
        X_train_scaled, y_train.values.astype(np.float32),
        epochs=50, batch_size=1024, validation_split=0.2, callbacks=[cb], verbose=1,
    )

test_df["pred_dnn"] = final_nn.predict(X_test_scaled).flatten()
_ = evaluate_model_performance(test_df[TARGET_COL], test_df["pred_dnn"], "Deep Neural Network")
_ = evaluate_portfolio(test_df, "pred_dnn")

## 6. Model 5 — AdaBoost

In [ ]:
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor

base_estimator = DecisionTreeRegressor(random_state=42)
adaboost = AdaBoostRegressor(estimator=base_estimator, random_state=42)
tscv = TimeSeriesSplit(n_splits=2)

print("Stage 1: Coarse search...")
coarse_grid = {
    "estimator__max_depth": [1, 2],
    "n_estimators": [10, 50, 100],
    "learning_rate": [0.001, 0.01, 0.1, 1.0],
}
search_coarse = GridSearchCV(adaboost, coarse_grid, cv=tscv, scoring="neg_mean_squared_error", n_jobs=-1)
search_coarse.fit(X_train, y_train)
print(f"Best coarse params: {search_coarse.best_params_}\n")

best = search_coarse.best_params_
print("Stage 2: Fine search...")
fine_grid = {
    "estimator__max_depth": [best["estimator__max_depth"]],
    "n_estimators": [max(1, best["n_estimators"] - 20), best["n_estimators"], best["n_estimators"] + 20],
    "learning_rate": [best["learning_rate"] * 0.5, best["learning_rate"], best["learning_rate"] * 2.0],
}
search_fine = GridSearchCV(adaboost, fine_grid, cv=tscv, scoring="neg_mean_squared_error", n_jobs=-1)
search_fine.fit(X_train, y_train)
print(f"Best fine params: {search_fine.best_params_}\n")

test_df["pred_adaboost"] = search_fine.best_estimator_.predict(X_test)
_ = evaluate_model_performance(test_df[TARGET_COL], test_df["pred_adaboost"], "AdaBoost")
_ = evaluate_portfolio(test_df, "pred_adaboost")

## 7. Model 6 — Maximum Sharpe Ratio Regression (MSRR)

Custom PyTorch loss that directly maximizes the portfolio's Sharpe ratio
instead of minimizing MSE.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

# Month indices for grouping
train_months = meta.loc[train_mask, "month_date"].astype("category").cat.codes.values
months_tensor = torch.tensor(train_months, dtype=torch.long)


class MaxSharpeRegression(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1, bias=True)

    def forward(self, x):
        return self.linear(x)


def sharpe_loss(predictions, targets, months):
    strategy_returns = predictions * targets
    unique_months = torch.unique(months)
    monthly_rets = torch.stack([
        strategy_returns[months == m].mean() for m in unique_months
    ])
    mean_ret = torch.mean(monthly_rets)
    std_ret = torch.std(monthly_rets) + 1e-6
    sharpe = (mean_ret / std_ret) * torch.sqrt(torch.tensor(12.0))
    return -sharpe  # minimize negative Sharpe


msrr_model = MaxSharpeRegression(X_train_tensor.shape[1])
optimizer = optim.Adam(msrr_model.parameters(), lr=0.01)

print("Training MSRR...")
for epoch in range(50):
    msrr_model.train()
    optimizer.zero_grad()
    preds = msrr_model(X_train_tensor)
    loss = sharpe_loss(preds, y_train_tensor, months_tensor)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1}/50 | Neg Sharpe: {loss.item():.4f}")

msrr_model.eval()
with torch.no_grad():
    test_df["pred_msrr"] = msrr_model(X_test_tensor).numpy().flatten()

_ = evaluate_model_performance(test_df[TARGET_COL], test_df["pred_msrr"], "MSRR")
_ = evaluate_portfolio(test_df, "pred_msrr")

## 8. Model 7 — IPCA (Instrumented PCA)

In [ ]:
# pip install ipca
from ipca import InstrumentedPCA

# Rebuild dataframes with proper MultiIndex for ipca library
df_model = pd.concat([meta, X_all], axis=1)
train_model = df_model.loc[train_mask].copy()
test_model = df_model.loc[test_mask].copy()

X_train_ipca = train_model[["id", "month_date"] + features].set_index(["id", "month_date"])
y_train_ipca = train_model.set_index(["id", "month_date"])[TARGET_COL]
X_test_ipca = test_model[["id", "month_date"] + features].set_index(["id", "month_date"])

ipca_model = InstrumentedPCA(n_factors=3, intercept=True)
print("Training IPCA...")
ipca_model.fit(X_train_ipca, y_train_ipca)

pred_ipca = ipca_model.predict(X=X_test_ipca, mean_factor=True)
test_df["pred_ipca"] = pred_ipca.values if hasattr(pred_ipca, "values") else pred_ipca

_ = evaluate_model_performance(test_df[TARGET_COL], test_df["pred_ipca"], "IPCA")
_ = evaluate_portfolio(test_df, "pred_ipca")

## 9. IPCA Factor Analysis

In [ ]:
# Gamma matrix: weights of each characteristic in factor construction
num_components = ipca_model.Gamma.shape[1]
col_names = (
    ["Intercept"] + [f"Factor_{i+1}" for i in range(ipca_model.n_factors)]
    if num_components > ipca_model.n_factors
    else [f"Factor_{i+1}" for i in range(ipca_model.n_factors)]
)

gamma_df = pd.DataFrame(ipca_model.Gamma, index=features, columns=col_names)
print("IPCA Gamma Matrix:")
display(gamma_df.style.background_gradient(cmap="coolwarm"))

# Tangency portfolio of IPCA factors
if ipca_model.Factors is not None:
    factor_rets = pd.DataFrame(ipca_model.Factors.T, columns=col_names)
    mean_factors = factor_rets.mean()
    cov_factors = factor_rets.cov()
    w_tangency = np.linalg.pinv(cov_factors).dot(mean_factors)
    w_tangency = w_tangency / np.sum(np.abs(w_tangency))

    print("\nTangency Portfolio Weights:")
    for name, w in zip(col_names, w_tangency):
        print(f"  {name}: {w:.2%}")

    tangency_returns = factor_rets.dot(w_tangency)
    ann_ret = tangency_returns.mean() * 12
    ann_vol = tangency_returns.std() * np.sqrt(12)
    print(f"\nIn-Sample Tangency: Return={ann_ret:.2%}, Vol={ann_vol:.2%}, Sharpe={ann_ret/ann_vol:.2f}")